# 🛒 Customer Segmentation Analysis
### By Animesh Ruhela | Data Analyst Project
---
**Objective:** Segment customers based on purchase behavior using Python and SQL to help businesses target the right audience and improve marketing strategies.

**Tools Used:** Python, SQL (SQLite), Pandas, Matplotlib, Seaborn, Scikit-learn

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print('✅ All Libraries Imported Successfully!')

## 📊 Step 2: Generate & Load Dataset

In [ ]:
# Generate Synthetic Customer Dataset
np.random.seed(42)
n = 500

data = {
    'CustomerID': range(1, n+1),
    'Age': np.random.randint(18, 70, n),
    'Annual_Income_k': np.random.randint(20, 150, n),
    'Spending_Score': np.random.randint(1, 100, n),
    'Total_Purchases': np.random.randint(1, 50, n),
    'Avg_Order_Value': np.round(np.random.uniform(100, 5000, n), 2),
    'Days_Since_Last_Purchase': np.random.randint(1, 365, n),
    'Gender': np.random.choice(['Male', 'Female'], n)
}

df = pd.DataFrame(data)
df.to_csv('customer_data.csv', index=False)
print(f'✅ Dataset Created: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)

## 🗄️ Step 3: SQL — Load Data into SQLite Database

In [ ]:
# Create SQLite Database
conn = sqlite3.connect('customer_segmentation.db')
df.to_sql('customers', conn, if_exists='replace', index=False)
print('✅ Data Loaded into SQLite Database!')

# SQL Query 1 — Basic Overview
query1 = '''
SELECT 
    COUNT(*) AS Total_Customers,
    ROUND(AVG(Age), 1) AS Avg_Age,
    ROUND(AVG(Annual_Income_k), 1) AS Avg_Income,
    ROUND(AVG(Spending_Score), 1) AS Avg_Spending_Score,
    ROUND(AVG(Avg_Order_Value), 2) AS Avg_Order_Value
FROM customers
'''
print('\n📋 SQL Query 1 — Customer Overview:')
pd.read_sql(query1, conn)

In [ ]:
# SQL Query 2 — Gender-wise Analysis
query2 = '''
SELECT 
    Gender,
    COUNT(*) AS Total_Customers,
    ROUND(AVG(Spending_Score), 1) AS Avg_Spending,
    ROUND(AVG(Annual_Income_k), 1) AS Avg_Income,
    ROUND(AVG(Avg_Order_Value), 2) AS Avg_Order_Value
FROM customers
GROUP BY Gender
'''
print('📋 SQL Query 2 — Gender-wise Analysis:')
pd.read_sql(query2, conn)

In [ ]:
# SQL Query 3 — High Value Customers (Top Spenders)
query3 = '''
SELECT CustomerID, Age, Annual_Income_k, Spending_Score, Avg_Order_Value
FROM customers
WHERE Spending_Score > 75 AND Annual_Income_k > 80
ORDER BY Avg_Order_Value DESC
LIMIT 10
'''
print('📋 SQL Query 3 — Top 10 High Value Customers:')
pd.read_sql(query3, conn)

In [ ]:
# SQL Query 4 — Age Group Segmentation
query4 = '''
SELECT 
    CASE 
        WHEN Age BETWEEN 18 AND 30 THEN 'Young (18-30)'
        WHEN Age BETWEEN 31 AND 45 THEN 'Middle-Aged (31-45)'
        WHEN Age BETWEEN 46 AND 60 THEN 'Senior (46-60)'
        ELSE 'Elder (60+)'
    END AS Age_Group,
    COUNT(*) AS Customers,
    ROUND(AVG(Spending_Score), 1) AS Avg_Spending,
    ROUND(AVG(Annual_Income_k), 1) AS Avg_Income
FROM customers
GROUP BY Age_Group
ORDER BY Avg_Spending DESC
'''
print('📋 SQL Query 4 — Age Group Analysis:')
pd.read_sql(query4, conn)

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Dataset Info
print('📊 Dataset Shape:', df.shape)
print('\n📋 Data Types:')
print(df.dtypes)
print('\n✅ Missing Values:')
print(df.isnull().sum())
print('\n📈 Statistical Summary:')
df.describe().round(2)

In [ ]:
# Distribution Plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Customer Data Distribution', fontsize=16, fontweight='bold')

cols = ['Age', 'Annual_Income_k', 'Spending_Score', 'Total_Purchases', 'Avg_Order_Value', 'Days_Since_Last_Purchase']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']

for i, (col, color) in enumerate(zip(cols, colors)):
    ax = axes[i//3][i%3]
    ax.hist(df[col], bins=20, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(col.replace('_', ' '), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('distribution_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution Plots Saved!')

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
numeric_cols = df.select_dtypes(include=np.number).drop('CustomerID', axis=1)
corr = numeric_cols.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Customer Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation Heatmap Saved!')

In [ ]:
# Income vs Spending Score Scatter
plt.figure(figsize=(10, 6))
colors_gender = {'Male': '#3498db', 'Female': '#e74c3c'}
for gender, color in colors_gender.items():
    subset = df[df['Gender'] == gender]
    plt.scatter(subset['Annual_Income_k'], subset['Spending_Score'],
                c=color, label=gender, alpha=0.6, s=60)
plt.xlabel('Annual Income (k)', fontsize=12)
plt.ylabel('Spending Score', fontsize=12)
plt.title('Annual Income vs Spending Score by Gender', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('income_vs_spending.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Step 5: K-Means Customer Segmentation

In [ ]:
# Feature Selection & Scaling
features = ['Age', 'Annual_Income_k', 'Spending_Score', 'Total_Purchases', 'Avg_Order_Value']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('✅ Features Scaled Successfully!')
print('Features Used:', features)

In [ ]:
# Elbow Method — Find Optimal Clusters
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method — Optimal K', fontsize=13, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', label='Optimal K=4')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score per K', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Best Silhouette Score: {max(silhouette_scores):.3f} at K={silhouette_scores.index(max(silhouette_scores))+2}')

In [ ]:
# Final K-Means Model with K=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

# Assign Meaningful Segment Names
cluster_names = {
    0: '💎 Premium Customers',
    1: '🛍️ Regular Shoppers',
    2: '💤 Inactive Customers',
    3: '🌱 New / Budget Customers'
}
df['Segment'] = df['Cluster'].map(cluster_names)
print('✅ Customer Segments Created!')
print('\n📊 Segment Distribution:')
print(df['Segment'].value_counts())

In [ ]:
# Segment Analysis
segment_analysis = df.groupby('Segment')[features].mean().round(2)
print('📋 Average Stats Per Segment:')
segment_analysis

In [ ]:
# Cluster Visualization — Income vs Spending
plt.figure(figsize=(12, 7))
colors_cluster = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for cluster_id in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster_id]
    plt.scatter(subset['Annual_Income_k'], subset['Spending_Score'],
                c=colors_cluster[cluster_id],
                label=cluster_names[cluster_id],
                alpha=0.7, s=70)

plt.xlabel('Annual Income (k)', fontsize=13)
plt.ylabel('Spending Score', fontsize=13)
plt.title('Customer Segmentation — Income vs Spending Score', fontsize=15, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Segmentation Plot Saved!')

In [ ]:
# Segment Distribution Pie Chart
segment_counts = df['Segment'].value_counts()
plt.figure(figsize=(9, 7))
plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%',
        colors=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'],
        startangle=140, explode=[0.05]*4)
plt.title('Customer Segment Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('segment_pie_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 🗄️ Step 6: Save Results Back to SQL

In [ ]:
# Save Segmented Data to SQLite
df.to_sql('customer_segments', conn, if_exists='replace', index=False)

# SQL Query — Segment Summary
query_final = '''
SELECT 
    Segment,
    COUNT(*) AS Total_Customers,
    ROUND(AVG(Annual_Income_k), 1) AS Avg_Income,
    ROUND(AVG(Spending_Score), 1) AS Avg_Spending,
    ROUND(AVG(Avg_Order_Value), 2) AS Avg_Order_Value,
    ROUND(AVG(Total_Purchases), 1) AS Avg_Purchases
FROM customer_segments
GROUP BY Segment
ORDER BY Avg_Spending DESC
'''
print('📋 Final SQL Summary — All Segments:')
result = pd.read_sql(query_final, conn)
print(result.to_string(index=False))
conn.close()
print('\n✅ Results Saved to Database!')

## 📝 Step 7: Conclusions & Business Insights

In [ ]:
print("""
====================================================
📊 CUSTOMER SEGMENTATION — KEY INSIGHTS
====================================================

💎 Premium Customers:
   → High income, high spending score
   → Focus: Loyalty programs, exclusive offers
   → Priority: RETAIN at all costs

🛍️ Regular Shoppers:
   → Medium income, medium spending
   → Focus: Upsell & cross-sell campaigns
   → Priority: GROW their basket size

💤 Inactive Customers:
   → Have income but not spending
   → Focus: Re-engagement campaigns, discounts
   → Priority: WIN BACK strategy

🌱 New / Budget Customers:
   → Low income, low spending
   → Focus: Budget offers, entry-level products
   → Priority: NURTURE & convert

====================================================
✅ Project by: Animesh Ruhela
✅ Tools: Python, SQL, Pandas, Scikit-learn, Seaborn
====================================================
""")